In [ ]:
import json

import agama
import gizmo_analysis as gizmo
import h5py
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import numpyro.distributions as dist
import pandas as pd
import seaborn as sns
from chainconsumer import Chain, ChainConsumer, PlotConfig
from numpyro.infer import MCMC, NUTS
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

agama.setUnits(mass=1, length=1, velocity=1)
units = agama.getUnits()

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 10
MEDIUM_SIZE = 10
BIGGER_SIZE = 10

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
# sim_dir = "/Users/z5114326/Documents/simulations/"
sim_dir = "/Volumes/Expansion/simulations/"

sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]

sim_codes = sim_dir + "simulation_codes.json"
with open(sim_codes) as sim_json:
    sim_data = json.load(sim_json)

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}

Rd_json = "../data/sim_data/Rd_dict.json"
with open(Rd_json, "r") as f:
    Rd_dict = json.load(f)

In [ ]:
# From Anake / JBH + G 16

Rd_mw = 2.6
Rsol_mw = 8.2  # kpc

r_lim = 35
rbins = np.linspace(0, r_lim, 100)
rmid = 0.5 * (rbins[1:] + rbins[:-1])

In [ ]:
fn0_json = "../data/mw_gcs/fno_mw_gcs.json"
with open(fn0_json, "r") as f:
    fn0_dict = json.load(f)

xlogfit = np.arange(4, 7.1, 0.1)
A = fn0_dict["A"]
B = fn0_dict["B"]
C = fn0_dict["C"]

In [ ]:
halo_dict = {
    sim: {
        "q": None,
        "p": None,
        "rmid": None,
        "rho_halo": None,
        "log_rho_sol": {"lower": None, "center": None, "upper": None},
        "alpha": {"lower": None, "center": None, "upper": None},
        "beta": {"lower": None, "center": None, "upper": None},
    }
    for sim in sim_lst
}

In [ ]:
def get_circ(pxyz, vxyz, pot_nbody):
    af = agama.ActionFinder(pot_nbody, interp=False)

    ep = pot_nbody.potential(pxyz)
    ek = 0.5 * np.sum(vxyz**2, axis=1)
    et = ek + ep

    ic = np.hstack((pxyz, vxyz))
    ioms = af(ic)
    lz = ioms[:, 2]

    r_circs = pot_nbody.Rcirc(E=et)
    xyz = np.column_stack((r_circs, r_circs * 0, r_circs * 0))
    v_circs = np.sqrt(-r_circs * pot_nbody.force(xyz)[:, 0])
    vel = np.column_stack((v_circs * 0, v_circs, v_circs * 0))
    ic_circ = np.concatenate((xyz, vel), axis=1)
    lz_circ = af(ic_circ)[:, 2]

    circ = lz / lz_circ

    E0 = pot_nbody.potential((0, 0, 0))
    etn = et / np.abs(E0)

    radii_pos = pot_nbody.Rperiapo(ic)
    r_per = radii_pos[:, 0]
    r_apo = radii_pos[:, 1]

    ecc = (r_apo - r_per) / (r_apo + r_per)

    return circ, etn, ecc


def rho_R(mask, z, zslice, R, Rbins, mass_s):

    msk = mask & (np.abs(z) < zslice)

    M, _ = np.histogram(R[msk], bins=Rbins, weights=mass_s[msk])

    vol = np.pi * (Rbins[1:] ** 2 - Rbins[:-1] ** 2) * (2 * zslice)

    rho = M / vol
    rho[rho <= 0] = np.nan

    return rho


def rho_z(mask, z, zbins, R, Rmax, mass_s):

    msk = mask & (R < Rmax)

    M, _ = np.histogram(z[msk], bins=zbins, weights=mass_s[msk])

    vol = np.pi * Rmax**2 * (zbins[1:] - zbins[:-1])

    rho = M / vol
    rho[rho <= 0] = np.nan

    return rho


def beta_anisotropy(vsph):
    vr = vsph[:, 0]
    vtheta = vsph[:, 1]
    vphi = vsph[:, 2]

    sr2 = np.var(vr)
    st2 = np.var(vtheta)
    sp2 = np.var(vphi)

    return 1.0 - (st2 + sp2) / (2.0 * sr2)


def cartesian_to_spherical_velocity(pxyz, vxyz):
    """
    Convert Cartesian velocities to spherical velocity components.

    Parameters
    ----------
    pxyz : (N, 3) array
        Cartesian positions [x, y, z].
    vxyz : (N, 3) array
        Cartesian velocities [vx, vy, vz].

    Returns
    -------
    vsph : (N, 3) array
        Spherical velocity components [v_r, v_theta, v_phi].
    """

    x, y, z = pxyz.T
    vx, vy, vz = vxyz.T

    r = np.sqrt(x**2 + y**2 + z**2)
    R = np.sqrt(x**2 + y**2)

    # avoid divide-by-zero
    eps = np.finfo(float).eps
    r = np.maximum(r, eps)
    R = np.maximum(R, eps)

    er = np.column_stack(
        [
            x / r,
            y / r,
            z / r,
        ]
    )

    etheta = np.column_stack(
        [
            x * z / (r * R),
            y * z / (r * R),
            -R / r,
        ]
    )

    ephi = np.column_stack(
        [
            -y / R,
            x / R,
            np.zeros_like(R),
        ]
    )

    vr = np.sum(vxyz * er, axis=1)
    vtheta = np.sum(vxyz * etheta, axis=1)
    vphi = np.sum(vxyz * ephi, axis=1)

    return np.column_stack([vr, vtheta, vphi])


def halo_model(r, Rsol, logrho_obs=None, sigma=None):

    logrho_sol = numpyro.sample("logrho_sol", dist.Uniform(3, 10))
    alpha = numpyro.sample("alpha", dist.Uniform(0.0, 5.0))
    beta = numpyro.sample("beta", dist.Uniform(0.0, 0.5))
    mu = logrho_sol - alpha * jnp.log10(r / Rsol) - beta * r / jnp.log(10.0)

    numpyro.sample("obs", dist.Normal(mu, sigma), obs=logrho_obs)


def find_representatibe_it_id(ghost_data):
    iter_stats = []
    for it_id in ghost_data.keys():
        src_dat = ghost_data[it_id]["source"]

        log_mfor = src_dat["logm_tfor"][()]
        mfor = 10**log_mfor
        mfor_ev = 0.55 * mfor
        mz0 = np.array([0 if m0 == -1 else 10**m0 for m0 in src_dat["logm_tz0"][()]])
        mdifs = mfor_ev - mz0

        log_fNO_mean = A + B * log_mfor + C * log_mfor**2
        fNO_mean = 10**log_fNO_mean
        gc_mass = (1 - fNO_mean) * mdifs

        gc_feh = src_dat["feh"][()]

        gc_r = np.linalg.norm(src_dat["pxyz_snap600"][()], axis=1)

        iter_stats.append(
            {
                "it": it_id,
                "n_gc": len(gc_mass),
                "mass_mean": np.mean(gc_mass),
                "mass_med": np.median(gc_mass),
                "mass_tot": np.sum(gc_mass),
                "feh_mean": np.mean(gc_feh),
                "feh_med": np.median(gc_feh),
                "r_mean": np.mean(gc_r),
                "r_med": np.median(gc_r),
            }
        )

    df = pd.DataFrame(iter_stats)

    mean_stats = df.drop(columns=["it"]).mean()

    dist = np.zeros(len(df))
    for col in mean_stats.index:
        dist += ((df[col] - mean_stats[col]) / df[col].std()) ** 2
    rep_it = df.iloc[np.argmin(dist)]["it"]

    print("Representative iteration:", rep_it)
    return rep_it

In [ ]:
for sim in sim_lst:
    fire_dir = sim_dir + sim + "/" + sim + "_res7100"

    pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (600, 600)
    pot_nbody = agama.Potential(pot_file)

    # part = gc_utils.open_snapshot(600, fire_dir, ["star"])
    part = gizmo.io.Read.read_snapshots("star", "index", 600, fire_dir, assign_hosts_rotation=True)

    #####################################################################################

    # rmsk = part["star"].prop("host.distance.principal.total") < r_lim  # so can properly sample the halo

    # pxyz = part["star"].prop("host.distance.principal", rmsk)
    # vxyz = part["star"].prop("host.velocity.principal", rmsk)

    # mass = part["star"]["mass"][rmsk]
    # fehs = part["star"].prop("metallicity.fe", rmsk)

    pxyz = part["star"].prop("host.distance.principal")
    vxyz = part["star"].prop("host.velocity.principal")

    mass = part["star"]["mass"]
    fehs = part["star"].prop("metallicity.fe")

    # GC correction ###################################

    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    # for ease of calculation find the iteration which best represents the ensemble
    it_id = find_representatibe_it_id(ghost_data)
    src_dat = ghost_data[it_id]["source"]

    log_mfor = src_dat["logm_tfor"][()]
    mfor = 10**log_mfor
    mfor_ev = 0.55 * mfor
    mz0 = np.array([0 if m0 == -1 else 10**m0 for m0 in src_dat["logm_tz0"][()]])
    mdifs = mfor_ev - mz0

    # BK23 relation
    log_fNO_mean = A + B * log_mfor + C * log_mfor**2
    fNO_mean = 10**log_fNO_mean
    gc_mass = (1 - fNO_mean) * mdifs

    gc_feh = src_dat["feh"][()]
    pidxs_600 = src_dat["pidx_snap600"][()]
    ptypes = np.array([ptype.decode("utf-8") for ptype in src_dat["ptype"][()]])
    gc_smsk = ptypes == "star"

    gc_pxyz = src_dat["pxyz_snap600"][()]
    gc_vxyz = src_dat["vxyz_snap600"][()]

    # replace star tracer particle masses and fehs with those of gc properties

    mass[pidxs_600[gc_smsk]] = gc_mass[gc_smsk]
    fehs[pidxs_600[gc_smsk]] = gc_feh[gc_smsk]
    pxyz[pidxs_600[gc_smsk]] = gc_pxyz[gc_smsk]
    vxyz[pidxs_600[gc_smsk]] = gc_vxyz[gc_smsk]

    len0 = len(mass)

    # append dm tracer particle gc masses and fehs

    mass = np.append(mass, gc_mass[~gc_smsk])
    fehs = np.append(fehs, gc_feh[~gc_smsk])
    pxyz = np.concatenate([pxyz, gc_pxyz[~gc_smsk]], axis=0)
    vxyz = np.concatenate([vxyz, gc_vxyz[~gc_smsk]], axis=0)

    len1 = len(mass)

    # quick check to ensure it all worked
    check = (len1 - len0) == np.sum(ptypes == "dark")
    assert check, "GC assignment issue"

    # mask ############################################

    rmsk = np.linalg.norm(pxyz, axis=1) < r_lim

    pxyz = pxyz[rmsk]
    vxyz = vxyz[rmsk]
    mass = mass[rmsk]
    fehs = fehs[rmsk]

    ###################################################

    vsph = cartesian_to_spherical_velocity(pxyz, vxyz)

    circ, etn, ecc = get_circ(pxyz, vxyz, pot_nbody)
    sph = circ < 0.2

    pxyz_s = pxyz[sph]
    mass_s = mass[sph]

    # Clustering ##################################################################

    etn_s = etn[sph]
    feh_s = fehs[sph]
    ecc_s = ecc[sph]

    X = np.column_stack([etn_s, ecc_s, feh_s])

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    labels = KMeans(n_clusters=2, n_init=20, random_state=42).fit_predict(Xs)

    # Remove noise particles
    clusters = np.unique(labels)
    clusters = clusters[clusters >= 0]

    # Mean binding energy of each cluster
    mu_etn = np.array([etn_s[labels == c].mean() for c in clusters])

    # Most bound cluster -> bulge
    bulge_cluster = clusters[np.argmin(mu_etn)]

    # Everything else -> halo
    bulge = labels == bulge_cluster
    halo = np.isin(labels, clusters[clusters != bulge_cluster])

    # Optional: ignore noise particles entirely
    valid = labels >= 0

    bulge &= valid
    halo &= valid

    # Density Profiles ##############################################################

    x = pxyz_s[:, 0]
    y = pxyz_s[:, 1]
    z = pxyz_s[:, 2]

    R = np.sqrt(x**2 + y**2)

    Rbins = np.linspace(0, 30, 60)
    Rcen = 0.5 * (Rbins[1:] + Rbins[:-1])

    zbins = np.linspace(-20, 20, 80)
    zcen = 0.5 * (zbins[1:] + zbins[:-1])

    Rmax = 30.0
    zslice = 0.5

    # Radial #################
    rhoR_sph = rho_R(np.ones(len(mass_s), dtype=bool), z, zslice, R, Rbins, mass_s)
    rhoR_bulge = rho_R(bulge, z, zslice, R, Rbins, mass_s)
    rhoR_halo = rho_R(halo, z, zslice, R, Rbins, mass_s)

    # Vertical #################
    rhoz_sph = rho_z(np.ones(len(mass_s), dtype=bool), z, zbins, R, Rmax, mass_s)
    rhoz_bulge = rho_z(bulge, z, zbins, R, Rmax, mass_s)
    rhoz_halo = rho_z(halo, z, zbins, R, Rmax, mass_s)

    vsph_s = vsph[sph]
    vsph_b = vsph_s[bulge]
    vsph_h = vsph_s[halo]

    beta_bulge = beta_anisotropy(vsph_b)
    beta_halo = beta_anisotropy(vsph_h)

    # Plotting ###########################################################

    fig, ax = plt.subplots(3, 4, figsize=(14, 10))

    # subsample for plotting
    nplot = 50000
    idx = np.random.choice(len(X), nplot, replace=False)

    ib = np.random.choice(np.where(bulge[idx])[0], min(nplot, bulge[idx].sum()), replace=False)

    ih = np.random.choice(np.where(halo[idx])[0], min(nplot, halo[idx].sum()), replace=False)

    # bulge
    Eb = etn_s[idx][ib]
    eb = ecc_s[idx][ib]
    Fb = feh_s[idx][ib]

    # halo
    Eh = etn_s[idx][ih]
    eh = ecc_s[idx][ih]
    Fh = feh_s[idx][ih]

    vars_b = [Eb, eb, Fb]
    vars_h = [Eh, eh, Fh]

    labels = [r"$E/|E_0|$", r"$e$", r"$\mathrm{[Fe/H]}$"]
    lims = [(-1, 0), (0, 1), (-4, 1)]

    for i in range(3):
        for j in range(3):
            pax = ax[i, j]

            if i < j:
                pax.axis("off")
                continue

            if i == j:
                pax.hist(vars_h[i], bins=60, density=True, histtype="step", color="b", lw=2)

                pax.hist(vars_b[i], bins=60, density=True, histtype="step", color="r", lw=2)

                pax.set_xlim(lims[i])

            else:
                sns.kdeplot(
                    x=vars_h[j], y=vars_h[i], levels=(0.393, 0.865, 0.989), color="b", fill=False, ax=pax
                )

                sns.kdeplot(
                    x=vars_b[j], y=vars_b[i], levels=(0.393, 0.865, 0.989), color="r", fill=False, ax=pax
                )

                pax.set_xlim(lims[j])
                pax.set_ylim(lims[i])

            if i == 2:
                pax.set_xlabel(labels[j])
            else:
                pax.set_xlabel("")

            if j == 0 and i > 0:
                pax.set_ylabel(labels[i])
            else:
                pax.set_ylabel("")

    # ============================================================
    # DENSITY PROFILES (right-most column)
    # ============================================================

    # Radial density profile
    ax[0, 3].semilogy(Rcen, rhoR_sph, "k--", lw=3, label="spheroid")
    ax[0, 3].semilogy(Rcen, rhoR_bulge, "r", lw=2, label="bulge")
    ax[0, 3].semilogy(Rcen, rhoR_halo, "b", lw=2, label="halo")

    ax[0, 3].set_xlabel("R [kpc]")
    ax[0, 3].set_ylabel(r"$\rho(R,z\simeq0)$")
    ax[0, 3].legend(fontsize=8)

    # Vertical density profile
    ax[1, 3].semilogy(zcen, rhoz_sph, "k--", lw=3, label="spheroid")
    ax[1, 3].semilogy(zcen, rhoz_bulge, "r", lw=2, label="bulge")
    ax[1, 3].semilogy(zcen, rhoz_halo, "b", lw=2, label="halo")

    ax[1, 3].set_xlabel("z [kpc]")
    ax[1, 3].set_ylabel(r"$\rho(z)$")

    text = (
        rf"$\beta_{{\rm Bulge}}={beta_bulge:.2f}$"
        "\n"
        rf"$\beta_{{\rm Halo}}={beta_halo:.2f}$"
    )

    ax[1, 3].text(
        0.05,
        0.80,
        text,
        transform=ax[1, 3].transAxes,
        fontsize=10,
        bbox=dict(facecolor="white", edgecolor="none"),
    )

    ax[0, 1].text(
        0.5,
        0.5,
        sim,
        transform=ax[0, 1].transAxes,
        fontsize=50,
        bbox=dict(facecolor="white", edgecolor="none"),
        ha="center",
        va="center",
    )

    # bottom-right unused
    ax[2, 3].axis("off")

    plt.tight_layout()

    decomp = "../data/figures/halo_fitting/" + sim + "_halo_decomposition.pdf"
    fig.savefig(decomp)

    plt.show()

    # Halo Rotation #####################################################################################

    # =====================================================
    # Halo particles
    # =====================================================

    pxyz_halo = pxyz_s[halo]
    mass_halo = mass_s[halo]

    x = pxyz_halo[:, 0]
    y = pxyz_halo[:, 1]
    z = pxyz_halo[:, 2]

    # =====================================================
    # Mass-weighted shape tensor
    # =====================================================

    S = np.array(
        [
            [
                np.average(x * x, weights=mass_halo),
                np.average(x * y, weights=mass_halo),
                np.average(x * z, weights=mass_halo),
            ],
            [
                np.average(x * y, weights=mass_halo),
                np.average(y * y, weights=mass_halo),
                np.average(y * z, weights=mass_halo),
            ],
            [
                np.average(x * z, weights=mass_halo),
                np.average(y * z, weights=mass_halo),
                np.average(z * z, weights=mass_halo),
            ],
        ]
    )

    # =====================================================
    # Principal axes
    # =====================================================

    evals, evecs = np.linalg.eigh(S)

    # largest eigenvalue = major axis
    idx = np.argsort(evals)[::-1]

    evals = evals[idx]
    evecs = evecs[:, idx]

    # =====================================================
    # Intrinsic axis ratios
    # =====================================================

    p_halo = np.sqrt(evals[1] / evals[0])
    q_halo = np.sqrt(evals[2] / evals[0])

    print(f"p_halo = {p_halo:.3f}")
    print(f"q_halo = {q_halo:.3f}")

    # =====================================================
    # Rotate into principal-axis frame
    # =====================================================

    xyz_rot = pxyz_halo @ evecs

    X = xyz_rot[:, 0]  # major axis
    Y = xyz_rot[:, 1]  # intermediate axis
    Z = xyz_rot[:, 2]  # minor axis

    # =====================================================
    # Ellipsoidal radius
    # =====================================================

    r_e_halo = np.sqrt(X**2 + Y**2 / p_halo**2 + Z**2 / q_halo**2)

    halo_dict[sim]["q"] = q_halo
    halo_dict[sim]["p"] = p_halo

    # Halo Density Binning ###################################################################################

    mass_shell, _ = np.histogram(r_e_halo, bins=rbins, weights=mass_halo)
    vol_shell = 4.0 * np.pi / 3.0 * p_halo * q_halo * (rbins[1:] ** 3 - rbins[:-1] ** 3)
    rho_halo = mass_shell / vol_shell

    halo_dict[sim]["rmid"] = rmid
    halo_dict[sim]["rho_halo"] = rho_halo

    # Halo Density Fitting ###################################################################################

    plot_corners = True
    mask_r = True  # match the Horta's work and only fit to data outside of inner kpc.

    if mask_r:
        rmsk = rmid > 1
    else:
        rmsk = np.ones(len(rmid), dtype=bool)

    # =====================================================
    # Prepare profile for fitting
    # =====================================================

    good = np.isfinite(rho_halo[rmsk]) & (rho_halo[rmsk] > 0) & np.isfinite(rmid[rmsk]) & (rmid[rmsk] > 0)

    r = rmid[rmsk][good]

    logrho = np.log10(rho_halo[rmsk][good])

    good2 = np.isfinite(logrho)

    r = r[good2]
    logrho = logrho[good2]

    sigma_logrho = np.ones(len(logrho), dtype=float) * 0.05

    # =====================================================
    # Run MCMC
    # =====================================================

    nuts_kernel = NUTS(halo_model)

    mcmc = MCMC(
        nuts_kernel,
        num_warmup=1000,
        num_samples=10000,
    )

    Rsol = Rsol_mw * Rd_dict[sim] / Rd_mw

    mcmc.run(
        jax.random.PRNGKey(0),
        r=jnp.asarray(r, dtype=float),
        Rsol=float(Rsol),
        logrho_obs=jnp.asarray(logrho, dtype=float),
        sigma=jnp.asarray(sigma_logrho, dtype=float),
    )

    samples = mcmc.get_samples()

    df = pd.DataFrame(
        {
            "log_rho_sol": samples["logrho_sol"],
            "alpha": samples["alpha"],
            "beta": samples["beta"],
        }
    )
    CC = ChainConsumer()
    CC.add_chain(Chain(samples=df, name=sim))
    summ = CC.analysis.get_summary()[sim]

    halo_dict[sim]["log_rho_sol"]["lower"] = np.round(summ["log_rho_sol"].lower, 3)
    halo_dict[sim]["log_rho_sol"]["center"] = np.round(summ["log_rho_sol"].center, 3)
    halo_dict[sim]["log_rho_sol"]["upper"] = np.round(summ["log_rho_sol"].upper, 3)

    halo_dict[sim]["alpha"]["lower"] = np.round(summ["alpha"].lower, 3)
    halo_dict[sim]["alpha"]["center"] = np.round(summ["alpha"].center, 3)
    halo_dict[sim]["alpha"]["upper"] = np.round(summ["alpha"].upper, 3)

    halo_dict[sim]["beta"]["lower"] = np.round(summ["beta"].lower, 3)
    halo_dict[sim]["beta"]["center"] = np.round(summ["beta"].center, 3)
    halo_dict[sim]["beta"]["upper"] = np.round(summ["beta"].upper, 3)

    if plot_corners:
        plot_df = df.rename(
            columns={
                "log_rho_sol": r"$\log_{10}\rho_{\odot,\rm equiv}$",
                "alpha": r"$\alpha$",
                "beta": r"$\beta$",
            }
        )

        CC = ChainConsumer()
        CC.add_chain(Chain(samples=plot_df, name=sim))

        CC.set_plot_config(
            PlotConfig(
                diagonal_tick_labels=True,
                shade=True,
                shade_alpha=0.4,
                sigma2d=True,
                flip=False,
            )
        )

        fig = CC.plotter.plot(figsize=(7, 7))

        fitting = "../data/figures/halo_fitting/" + sim + "_halo_fitting.pdf"
        fig.savefig(fitting)

In [ ]:
def convert(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: convert(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert(v) for v in obj]
    return obj


halo_dict_json = convert(halo_dict)

save_json = "../data/sim_data/halo_dict.json"
with open(save_json, "w") as f:
    json.dump(halo_dict_json, f, indent=4)